# Reader note

This notebook is part of the `arXiv:2606.04091` reproduction workflow. It reproduces `Figures 9, 10 & 11`, plots illustrating the projected reach for a target ($CaWO_4$, $SiO_2$, $Al_2O_3$, $GaAs$) for different threshold energies and mediators in `standard prescription`.<br>
Set `DATA_ROOT` to the directory containing the generated HDF5 data. Old execution outputs are intentionally cleared for release.


In [ ]:
import script_helpers.script_reach as dm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import numpy as np

# Generated HDF5 data root. Override this without editing the notebook by setting
DATA_ROOT='/path/to/generated/data before starting Jupyter.'

In [ ]:
### Set the target material to plot (CaWO4 SiO2, Al2O3, GaAs)
Target = "CaWO4"

### Set the threshold values to plot (in eV)
new_threshold_list = [0.001, 0.02] # 1 meV and 20 meV

### Define mediator types to plot (light_hadrophilic, heavy_hadrophilic, light_dark_photon)
Mediators = ["light_dark_photon"] # Choose the mediator name you want to plot

### numerics used in calculations (standard, standard_q_parallel)
Numerics = "standard"

### Halo models used in calculations (SHM, TSA, EMP)
Mod = ["SHM", "TSA", "EMP"]

### Choose your fiducial halo model (SHM)
fid_mod = "SHM"

### Set the data directory path as per your local machine
# Current path is set to the publicly available data repository for the paper
data_dir = DATA_ROOT + f"/Manuscript_data/{Target}_projected_reach/files_QQQ"

In [ ]:
### Conservative Velocity parameters in standard prescription
V_0 = [220, 200, 280]
V_E = [232, 217, 246]
V_ESC = [544, 450, 600]

fid_velocity = ("220", "232", "544") # Central values of the velocity parameters in the standard prescription

### Aggressive Velocity parameters in standard prescription
# V_0 = [236, 238, 239]
# V_E = [248, 250, 251]
# V_ESC = [503, 528, 552]

# fid_velocity = ("238", "250", "528") # Central values of the velocity parameters in the rms-matching prescription

In [ ]:
### Generate file prefixes to be read and check for duplicates. Depends on file names of saved data
prefixes = dm.generate_file_prefixes(
    Target=Target,
    Mediators=Mediators,
    Numerics=Numerics,
    V_0=V_0,
    V_E=V_E,
    V_ESC=V_ESC,
    Mod=Mod
)

dm.check_for_duplicates(prefixes)

In [ ]:
### Read fiducial rate and cross-section(sigma)
fiducial_data = dm.read_fiducial_data_from_diff(
    data_dir=data_dir, 
    Target=Target,
    Mediators=Mediators,
    Numerics=Numerics,
    fid_velocity=fid_velocity,
    fid_mod=fid_mod,
    new_threshold_list_meV=new_threshold_list,
    time_index=0,
    use_run_threshold_index=0
)

In [ ]:
# Compute group extremes: maximum and minimum rates and sigmas for each group of files with the same mediator type 
# and halo model when velocity parameters are varied.
group_max_rate, group_min_rate, group_max_sigma, group_min_sigma = dm.compute_group_extremes_from_diff(
    grouped_prefixes=prefixes,
    data_dir=data_dir, 
    fiducial_data=fiducial_data,
    new_threshold_list_meV=new_threshold_list
)

In [ ]:
# Select the mediator you want to plot
mediator_key = Mediators[0]  # Choose the mediator name you want to plot if you list multiple mediators in the Mediators list. 

# Compute group uncertainties: Given the group extremes for the selected mediator and halo model, 
# find the difference between the maximum and minimum rates and sigmas for each threshold.
group_uncertainty = dm.compute_group_uncertainties(
    mediator_key=mediator_key,
    fiducial_data=fiducial_data,
    group_max_rate=group_max_rate,
    group_min_rate=group_min_rate,
    mod=Mod
)

In [ ]:
# Reading TSA and EMP data for the central values of the velocity paramters

### Prefix for TSA and EMP
base_prefix = f"{Target}_{mediator_key}_{Numerics}"
TSA_prefix = f"{base_prefix}_TSA_{fid_velocity[0]}_{fid_velocity[1]}_{fid_velocity[2]}"
EMP_prefix = f"{base_prefix}_EMP_{fid_velocity[0]}_{fid_velocity[1]}_{fid_velocity[2]}"

read_func = getattr(dm, f"read_data_{mediator_key}_from_diff", None)
TSA_data = read_func(TSA_prefix, data_dir, new_threshold_list_meV=new_threshold_list, time_index=0, use_run_threshold_index=0)
EMP_data = read_func(EMP_prefix, data_dir, new_threshold_list_meV=new_threshold_list, time_index=0, use_run_threshold_index=0)

model_data_dict = {
    "TSA": TSA_data,
    "EMP": EMP_data,
}

# Compute model uncertainties for the selected mediator and halo model
model_unc = dm.compute_model_uncertainties(
    mediator_key,
    fiducial_data,
    model_data_dict
)

In [ ]:
# Setup
legend_labels = {"SHM": "SHM", "TSA": "Tsallis", "EMP": "Empirical"}
linestyles = {"0.001" : "solid", "0.02": "dashed", "0.1": "dashdot"}

# color
model_colors = {"SHM": "#3A86FF", "TSA": "#FF006E", "EMP": "#8338EC"}

## matplotlib formatting
settings = {
    # 'figure.constrained_layout.use': True,
    # 'mathtext.fontset': 'stix',
    # 'font.family': 'STIXGeneral',
    # LaTeX-like fonts
    'mathtext.fontset': 'cm',
    # 'font.family': 'serif',
    # 'font.serif': ['Computer Modern Roman'],
    # 'mathtext.fontset': 'dejavuserif',
    'font.family': 'DejaVu Serif',
    'font.size':18,
    # 'axes.labelsize': 'large',
    'lines.markersize': 5,
    'axes.linewidth':2.0,
    'xtick.major.size':8.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':1.5,
    'xtick.minor.width':1.0,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':8.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':1.5,
    'ytick.minor.width':1.0,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'contour.linewidth':3.0,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
}

plt.rcParams.update(**settings) 


In [ ]:
# Build a dict: model_sigma_curves[model][threshold] = (dm_mass, sigma_1d)
model_sigma_curves = {}

for model_name, (DM_mass_m, thresholds_m, rates_2D_m, sigmas_2D_m) in model_data_dict.items():
    model_sigma_curves[model_name] = {}
    thresholds_m = np.array(thresholds_m)

    for th in fiducial_data[mediator_key]["thresholds"]:
        idx_arr = np.where(thresholds_m == th)[0]
        if len(idx_arr) == 0:
            print(f"Warning: threshold {th} not in {model_name} data, skipping.")
            continue
        th_idx = int(idx_arr[0])
        model_sigma_curves[model_name][th] = (DM_mass_m, sigmas_2D_m[th_idx])


In [ ]:
### Plotting reach with Uncertainties: multiple thresholds

thresholds_to_plot = new_threshold_list  # Use the thresholds you want to plot

if mediator_key in fiducial_data:
    all_thresholds = sorted(fiducial_data[mediator_key]["thresholds"])
    thresholds = [th for th in thresholds_to_plot if th in all_thresholds]
    n_th = len(thresholds)

    # Figure with 1 top + N bottom panels
    fig = plt.figure(figsize=(10, 3.8 + 2.2 * n_th))  # scale height with #thresholds
    gs = GridSpec(1 + n_th, 1, figure=fig, height_ratios=[3.0] + [1.4] * n_th)

    ax_top = fig.add_subplot(gs[0])

    # Create bottom axes (one per threshold)
    ax_unc_list = [fig.add_subplot(gs[i + 1]) for i in range(n_th)]

    # TOP PANEL: reach curves for selected thresholds
    for th in thresholds:
        fid_dm_mass = fiducial_data[mediator_key][th]["dm_mass"]
        fid_sigmas  = fiducial_data[mediator_key][th]["sigmas"]

        ax_top.plot(
            fid_dm_mass, fid_sigmas,
            lw=0.9, color=model_colors["SHM"], linestyle=linestyles[str(th)],
            label=fr"Fiducial, $E_{{th}}={th}$ meV"
        )

        for model in Mod:
            color = model_colors[model]
            max_sigma = group_max_sigma.get((mediator_key, model, th), None)
            min_sigma = group_min_sigma.get((mediator_key, model, th), None)
            if max_sigma is not None and min_sigma is not None:
                ax_top.fill_between(fid_dm_mass, min_sigma, max_sigma, alpha=0.15, color=color)

        for model in ["TSA", "EMP"]:
            if model in model_sigma_curves and th in model_sigma_curves[model]:
                dm_mass_m, sigma_1d = model_sigma_curves[model][th]
                ax_top.plot(
                    dm_mass_m, sigma_1d,
                    color=model_colors[model], linestyle=linestyles[str(th)], lw=1.0)

    # BOTTOM PANELS: uncertainty band for selected threshold
    for i, th in enumerate(thresholds):
        ax = ax_unc_list[i]
        fid_dm_mass = fiducial_data[mediator_key][th]["dm_mass"]
        fid_sigmas  = fiducial_data[mediator_key][th]["sigmas"]

        ax.plot(
            fid_dm_mass, fid_sigmas,
            lw=1.0, color=model_colors["SHM"], linestyle=linestyles[str(th)],
        )

        for model in Mod:
            data = group_uncertainty.get((mediator_key, model, th))
            if data is not None:
                ax.fill_between(
                    fid_dm_mass,
                    data["uncertainty_min"],
                    data["uncertainty_max"],
                    alpha=0.2,
                    color=model_colors[model],
                )

        for model in ["TSA", "EMP"]:
            key_model = (mediator_key, model, th)
            if key_model in model_unc:
                data = model_unc[key_model]
                dm_mass_m = data["dm_mass"]
                unc_model = data["rate_uncertainty"]
                ax.plot(
                    dm_mass_m, unc_model,
                    color=model_colors[model], lw=1.0,
                    linestyle=linestyles[str(th)], label=model
                )

        # Panel label for threshold
        ax.text(
            0.75, 0.80, fr"$\omega_{{\text{{th}}}}={th*10**3}$ meV",
            transform=ax.transAxes,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85)
        )

        # Formatting (bottom)
        ax.set_xscale("log")
        ax.set_ylim(-1.0, 1.0)
        ax.set_yticks([-0.8, -0.4, 0.0, 0.4, 0.8])
        ax.tick_params(axis="both", direction="in",
                       top=True, bottom=True, left=True, right=True)
        ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.4)
        ax.set_xlim(3e-3, 1e3)
        ax.set_xticks([1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3])

        if i < n_th - 1:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel(r"$m_\chi$ (MeV)", fontsize=22)

        ax.set_ylabel(r"$R/R_{\rm fid}-1$", fontsize=18)

    # Formatting (top)
    ax_top.set_xscale("log")
    ax_top.set_yscale("log")
    ax_top.set_xlim(3e-3, 1e3)
    ax_top.set_xticks([1e-2, 1e-1, 1e0, 1e1, 1e2])
    ax_top.tick_params(labelbottom=True)
    ax_top.set_ylim(1e-45, 1e-38)
    ax_top.set_yticks([1e-44, 1e-43, 1e-42, 1e-41, 1e-40, 1e-39, 1e-38])

    ax_top.tick_params(axis="both", direction="in",
                       top=True, bottom=True, left=True, right=True)
    ax_top.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.4)

    if mediator_key == "light_dark_photon":
        ax_top.set_ylabel(r"$\overline{\sigma}_e$ (cm$^2$)", fontsize=22)
    else:
        ax_top.set_ylabel(r"$\overline{\sigma}_n$ (cm$^2$)", fontsize=22)

    # Custom legend (top)
    custom_handles = [
        Line2D([0], [0], color=model_colors['SHM'], lw=2.0, label='SHM'),
        Line2D([0], [0], color=model_colors['TSA'], lw=2.0, linestyle='-', label='Tsallis'),
        Line2D([0], [0], color=model_colors['EMP'], lw=2.0, linestyle='-', label='Empirical'),
    ]
    ax_top.legend(
        handles=custom_handles,
        loc='center',
        bbox_to_anchor=(0.845, 0.24),
        bbox_transform=ax_top.transAxes,
        frameon=True,
    )

    plt.subplots_adjust(hspace=0.0)
    plt.show()